
# <span style="color:blue">Log Compaction in Apache Kafka</span>

# What is Log Compaction?

Log Compaction is a Kafka storage mechanism that keeps the latest value for each key while removing older versions of that same key over time.

### Simple Definition

```text
Log Compaction
=
Keep Latest Record Per Key
```

Instead of storing every historical version forever, Kafka eventually retains the most recent value for each key.

---

# Why Do We Need Log Compaction?

Some applications do not need complete history.

They need:

```text
Current State
```

Examples:

```text
Customer Profile

Account Balance

Product Price

Inventory Count
```

For these use cases, the latest value is more important than all previous values.

---

# Example

Messages:

```text
User1 → Arif

User1 → Mohammed Arif

User1 → M Arif

User1 → Mohammad Arif
```

After Log Compaction:

```text
User1 → Mohammad Arif
```

Latest value survives.

---

# What is Retention?

Retention determines how long Kafka keeps messages.

Kafka can delete data based on:

```text
Time
```

or

```text
Size
```

---

## Time-Based Retention

Example:

```text
Retention = 7 Days
```

After 7 days:

```text
Messages Deleted
```

---

## Size-Based Retention

Example:

```text
Retention = 100 GB
```

When storage exceeds the limit:

```text
Old Messages Deleted
```

---

## Example

Topic:

```text
Order1

Order2

Order3

Order4
```

Retention:

```text
7 Days
```

After retention expires:

```text
Everything Deleted
```

---

# Difference Between Retention and Log Compaction

## Retention

Retention removes records based on:

```text
Time

OR

Size
```

Example:

```text
Retention = 7 Days
```

After 7 days:

```text
All Messages Deleted
```

---

## Log Compaction

Log Compaction removes:

```text
Older Versions Of Same Key
```

and keeps:

```text
Latest Version
```

---

## Example

Before:

```text
User1 → Arif

User1 → Mohammed Arif

User1 → Mohammad Arif
```

Retention:

```text
After 7 Days

Everything Deleted
```

---

Compaction:

```text
User1 → Mohammad Arif
```

Latest state survives.

---

# Retention vs Log Compaction Comparison

## Retention

```text
Goal
=
Remove Old Data
```

```text
Based On
=
Time Or Size
```

```text
Keeps Latest State?
=
No
```

---

## Log Compaction

```text
Goal
=
Maintain Latest State
```

```text
Based On
=
Message Key
```

```text
Keeps Latest State?
=
Yes
```

---

# Key Requirement For Log Compaction ⭐

Log Compaction works using:

```text
Message Key
```

Kafka compares records having the same key.

Example:

```text
Key = User1
```

Messages:

```text
User1 → Arif

User1 → Mohammed Arif

User1 → Mohammad Arif
```

Kafka can identify:

```text
Same Key
```

and compact older values.

---

## Important Interview Point

```text
Log Compaction is meaningful only for keyed records.
```

Without keys:

```text
Kafka cannot determine
which records belong together.
```

---

# Before and After Compaction Diagram

## Before Compaction

```text
Customer Topic

User1 → Arif

User2 → Ali

User1 → Mohammed Arif

User2 → Ali Khan

User1 → Mohammad Arif
```

---

## After Compaction

```text
Customer Topic

User1 → Mohammad Arif

User2 → Ali Khan
```

Latest value for each key remains.

---

# Log Compaction Is Not Immediate ⭐

A common misconception:

```text
New Record Arrives
        ↓
Old Record Deleted Immediately
```

❌ Incorrect

Kafka performs compaction in the background.

---

## Actual Behavior

```text
New Record Arrives
        ↓
Record Written
        ↓
Compaction Process Runs Later
        ↓
Old Records Removed
```

---

## Interview Point

```text
Log Compaction is asynchronous
and does not happen immediately.
```

---

# Latest Record Survives ⭐

Kafka ensures that the latest version for a key remains available.

Example:

Before:

```text
User1 → Arif

User1 → Mohammed Arif

User1 → Mohammad Arif
```

After:

```text
User1 → Mohammad Arif
```

At least the latest record survives.

---

# Tombstone Records ⭐⭐⭐

## What is a Tombstone Record?

A tombstone record in Apache Kafka is a message with a valid key and a null value (value = null) used to permanently delete data from a compacted topic.

---

## Example

Message:

```text
Key   = User1

Value = null
```

Meaning:

```text
Delete User1
```

---

## Flow

Before:

```text
User1 → Mohammad Arif
```

Delete Event:

```text
User1 → null
```

After Compaction:

```text
User1 Removed
```

---

# State Reconstruction ⭐⭐⭐

One major benefit of compacted topics is:

```text
State Reconstruction
```

---

## What is State Reconstruction?

A new consumer can read a compacted topic and rebuild the latest state of all entities.

---

## Example

Current Topic State:

```text
User1 → Mohammad Arif

User2 → Ali Khan

User3 → John
```

A new consumer starts.

After reading the compacted topic:

```text
Consumer Knows Current State
```

without processing the complete historical history.

---

# CDC Example (Debezium) ⭐⭐⭐

Architecture:

```text
MySQL
   ↓
Debezium
   ↓
Kafka Compacted Topic
   ↓
Consumers
```

---

## Updates

Database Changes:

```text
User1 → Arif

User1 → Mohammed Arif

User1 → Mohammad Arif
```

Compacted Topic:

```text
User1 → Mohammad Arif
```

Only latest state is retained.

---

## Why Important For CDC?

Consumers can easily reconstruct the latest state of database rows.

---

# Real World Use Cases

## Customer Profiles

```text
Customer_ID
      ↓
Latest Profile
```

---

## Product Catalog

```text
Product_ID
      ↓
Latest Product Details
```

---

## Account Balance

```text
Account_ID
      ↓
Latest Balance
```

---

## Inventory Management

```text
Item_ID
      ↓
Latest Inventory Count
```

---

## CDC Pipelines

```text
Database
   ↓
Kafka
   ↓
Data Warehouse
```

---

# Benefits of Log Compaction

✅ Reduced Storage

✅ Latest Value Available

✅ Faster Recovery

✅ State Reconstruction

✅ Useful For CDC

✅ Useful For Kafka Streams

✅ Useful For Event Sourcing

✅ Efficient Long-Term Storage

---

# Architecture Diagram

```text
Before

User1 → Arif

User1 → Mohammed Arif

User1 → Mohammad Arif

User2 → Ali

User2 → Ali Khan


          ↓

    Log Compaction


          ↓


After

User1 → Mohammad Arif

User2 → Ali Khan
```

---

# Quick Revision

```text
Retention
=
Delete Data Based On Time Or Size
```

---

```text
Log Compaction
=
Keep Latest Value Per Key
```

---

```text
Compaction Requires Keys
```

---

```text
Latest Record Survives
```

---

```text
Compaction Is Asynchronous
```

---

```text
Tombstone Record
=
Key + Null Value
```

---

```text
CDC Frequently Uses Compacted Topics
```

---

# Most Important Interview Statement ⭐

```text
Log Compaction retains the latest value for each key while removing older versions of the same key over time.
```

---

# <span style="color:red">Log Compaction Interview Questions & Answers</span>

## Q1. What is Log Compaction?

Log Compaction is a Kafka storage mechanism that keeps the latest value for each key and removes older versions of the same key over time.

---

## Q2. Why Do We Need Log Compaction?

To maintain current state while reducing storage requirements.

---

## Q3. What is Retention?

Retention defines how long Kafka keeps messages before deleting them based on time or size.

---

## Q4. What is the Difference Between Retention and Log Compaction?

Retention:

```text
Deletes Data Based On Time Or Size
```

Compaction:

```text
Keeps Latest Value Per Key
```

---

## Q5. Does Log Compaction Delete All Messages?

❌ No

It keeps the latest value for each key.

---

## Q6. Does Log Compaction Work Without Keys?

❌ No

Log Compaction relies on message keys.

---

## Q7. Why Are Keys Important For Compaction?

Keys allow Kafka to identify records belonging to the same entity.

---

## Q8. What Happens To Older Versions Of A Key?

Kafka eventually removes them during compaction.

---

## Q9. Does Kafka Compact Records Immediately?

❌ No

Compaction runs asynchronously in the background.

---

## Q10. What Is Kept After Compaction?

The latest value for each key.

---

## Q11. What Is a Tombstone Record?

A tombstone record in Apache Kafka is a message with a valid key and a null value (value = null) used to permanently delete data from a compacted topic.

---

## Q12. Why Are Tombstone Records Used?

To indicate that a key should be deleted.

---

## Q13. What Happens After a Tombstone Record Is Compacted?

The key is removed from the compacted topic.

---

## Q14. What Is State Reconstruction?

Rebuilding the latest state of all entities by reading a compacted topic.

---

## Q15. Why Is State Reconstruction Important?

New consumers can rebuild the current system state without replaying all history.

---

## Q16. Why Is Log Compaction Useful For CDC?

Because it helps maintain the latest database state for each record.

---

## Q17. What Is a Typical CDC Architecture?

```text
MySQL
   ↓
Debezium
   ↓
Kafka Compacted Topic
   ↓
Consumers
```

---

## Q18. Can Retention And Compaction Be Used Together?

✅ Yes

Kafka topics can be configured with both policies.

---

## Q19. What Is The Main Benefit Of Compacted Topics?

The latest state of every key remains available.

---

## Q20. What Types Of Applications Commonly Use Compaction?

✅ CDC

✅ Kafka Streams

✅ State Stores

✅ Product Catalogs

✅ Customer Profiles

---

## Q21. Is Log Compaction A Storage Optimization Feature?

✅ Yes

It reduces storage by removing obsolete records.

---

## Q22. What Happens If The Same Key Appears Multiple Times?

Kafka keeps the most recent value for that key.

---

## Q23. What Is The Main Limitation Of Log Compaction?

Meaningful compaction requires message keys.

---

## Q24. Which Is Better: Retention Or Log Compaction?

Neither.

They solve different problems.

---

## Q25. Most Important Log Compaction Interview Answer?

```text
Log Compaction retains the latest value for each key while removing older versions of the same key over time.
```